# Project: 3D Reconstruction from 2D Images
### Studienarbeit - Final Notebook

## 1. Setup and Configuration

### 1.1 Control Switch

**Set `RUN_FULL_RECONSTRUCTION_PIPELINE = True`** to process the original `.tiff` files, run the entire reconstruction, and save the final 3D volume. 

**Set `RUN_FULL_RECONSTRUCTION_PIPELINE = False`** to skip the processing and instead load the previously saved `reconstructed_volume_512.npy` file for quick visualization.

In [ ]:
RUN_FULL_RECONSTRUCTION_PIPELINE = True

### 1.2 Mount Drive, Clone Repo, and Install Dependencies

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

repo_path = '/content/bohboh'

if not os.path.exists(repo_path):
  !git clone https://github.com/hazempgm/bohboh.git
  %cd {repo_path}
else:
  %cd {repo_path}
  !git pull

!git checkout version_3
!pip install tifffile matplotlib scikit-image tqdm ipywidgets plotly

### 1.3 Imports and Path Definitions

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, widgets
from skimage.transform import resize
import plotly.graph_objects as go

# Add the src directory to the Python path
sys.path.append(os.path.abspath('src'))

# Import our custom modules
from data_loader import load_tiff_images
from reconstruction import reconstruct_slice, reconstruct_full_volume

# Define paths
image_dir = '/content/drive/MyDrive/bahbouh/data_bohboh/Images'
output_dir = '/content/drive/MyDrive/bahbouh/'
volume_path = os.path.join(output_dir, 'reconstructed_volume_512.npy')

print("Setup Complete.")
print(f"Volume will be saved to or loaded from: {volume_path}")

---

## 2. Full Pipeline: From TIFF to 3D Volume

This section will only run if `RUN_FULL_RECONSTRUCTION_PIPELINE` is set to `True`.

In [ ]:
if RUN_FULL_RECONSTRUCTION_PIPELINE:
    # 2.1 Load raw TIFF images
    print("--- Step 2.1: Loading TIFF images ---")
    image_stack = load_tiff_images(image_dir)
    if image_stack.size > 0:
        print(f"Shape of the stack: {image_stack.shape}")
        
        # 2.2 Visualize a sample projection
        print("\n--- Step 2.2: Visualizing sample projection ---")
        middle_index = image_stack.shape[0] // 2
        plt.figure(figsize=(6, 6))
        plt.imshow(image_stack[middle_index], cmap='gray')
        plt.title(f'Sample Projection Image (Slice #{middle_index})')
        plt.show()
        
        # 2.3 Reconstruct and visualize a single test slice
        print("\n--- Step 2.3: Reconstructing and visualizing a test slice ---")
        slice_height_index = image_stack.shape[1] // 2
        sinogram = image_stack[:, slice_height_index, :]
        theta = np.linspace(0., 180., image_stack.shape[0], endpoint=False)
        
        plt.figure(figsize=(8, 4))
        plt.imshow(sinogram, cmap='gray', aspect='auto')
        plt.title('Test Sinogram')
        plt.xlabel('Detector Position'); plt.ylabel('Projection Angle')
        plt.show()
        
        reconstructed_test_slice = reconstruct_slice(sinogram, theta)
        plt.figure(figsize=(6, 6))
        plt.imshow(reconstructed_test_slice, cmap='gray')
        plt.title('Test Reconstructed Cross-Section')
        plt.show()

        # 2.4 Downsample for full reconstruction
        print("\n--- Step 2.4: Downsampling for full reconstruction ---")
        new_shape = (image_stack.shape[0], 512, 512)
        print(f"Downsampling from {image_stack.shape} to {new_shape}...")
        image_stack_small = resize(image_stack, new_shape, anti_aliasing=True)
        print(f"New shape: {image_stack_small.shape}")

        # 2.5 Run full volume reconstruction
        print("\n--- Step 2.5: Running full volume reconstruction ---")
        reconstructed_volume = reconstruct_full_volume(image_stack_small)
        print(f"Final reconstructed volume shape: {reconstructed_volume.shape}")

        # 2.6 Save the final volume
        print("\n--- Step 2.6: Saving final volume ---")
        print(f"Saving to: {volume_path}")
        np.save(volume_path, reconstructed_volume)
        print("File saved successfully.")
    else:
        print("Could not load TIFF images. Halting pipeline.")
        reconstructed_volume = None
else:
    print("Skipping full pipeline as requested.")
    reconstructed_volume = None # Ensures the variable exists

---

## 3. Load Volume and Visualize Slices

This section loads the 3D volume (either from the pipeline above or from the disk) and provides interactive 2D slice viewers.

In [ ]:
# If we didn't run the pipeline, load the volume from disk
if not RUN_FULL_RECONSTRUCTION_PIPELINE:
    if os.path.exists(volume_path):
        print(f"Loading existing volume from {volume_path}...")
        reconstructed_volume = np.load(volume_path)
        print("Volume loaded successfully.")
    else:
        print(f"ERROR: Saved volume not found at {volume_path}.")
        print("Set RUN_FULL_RECONSTRUCTION_PIPELINE = True and re-run to generate it.")
        reconstructed_volume = None

if reconstructed_volume is not None:
    print(f"\nVolume ready for visualization. Shape: {reconstructed_volume.shape}")
else:
    print("\nNo volume data to visualize.")

### 3.1 Interactive 2D Slice Viewers

In [ ]:
def view_slices_2d(volume):
    if volume is None:
        print("Volume not loaded. Cannot create 2D viewer.")
        return
    # This function is the same as before, providing the 2D slice views.
    def plot_axial_slice(slice_z): plt.figure(figsize=(7, 7)); plt.imshow(volume[slice_z, :, :], cmap='gray'); plt.title(f'Axial View (Slice Z = {slice_z})'); plt.show()
    def plot_coronal_slice(slice_y): plt.figure(figsize=(7, 7)); plt.imshow(volume[:, slice_y, :], cmap='gray'); plt.title(f'Coronal View (Slice Y = {slice_y})'); plt.show()
    def plot_sagittal_slice(slice_x): plt.figure(figsize=(7, 7)); plt.imshow(volume[:, :, slice_x], cmap='gray'); plt.title(f'Sagittal View (Slice X = {slice_x})'); plt.show()
    print("--- Axial Viewer (Top-Down) ---"); interact(plot_axial_slice, slice_z=widgets.IntSlider(min=0, max=volume.shape[0]-1, step=1, value=volume.shape[0]//2))
    print("\n--- Coronal Viewer (Front-Back) ---"); interact(plot_coronal_slice, slice_y=widgets.IntSlider(min=0, max=volume.shape[1]-1, step=1, value=volume.shape[1]//2))
    print("\n--- Sagittal Viewer (Left-Right) ---"); interact(plot_sagittal_slice, slice_x=widgets.IntSlider(min=0, max=volume.shape[2]-1, step=1, value=volume.shape[2]//2))

view_slices_2d(reconstructed_volume)

---

## 4. Interactive 3D Volume Rendering

This final section provides a true 3D visualization of the object that you can rotate and zoom using your mouse.

### 4.1 Downsample for Performance

Rendering a full `(512, 512, 512)` volume is very slow in a browser. We first downsample the volume to a smaller size like `(128, 128, 128)` to ensure the 3D plot is interactive and responsive.

In [ ]:
volume_for_3d_vis = None
if reconstructed_volume is not None:
    vis_shape = (128, 128, 128)
    print(f"Downsampling volume for 3D visualization to {vis_shape}...")
    volume_for_3d_vis = resize(reconstructed_volume, vis_shape, anti_aliasing=True)
    print("Downsampling complete.")
else:
    print("No volume data to prepare for 3D visualization.")

### 4.2 Create 3D Plot

In [ ]:
if volume_for_3d_vis is not None:
    Z, Y, X = volume_for_3d_vis.shape
    x, y, z = np.mgrid[:X, :Y, :Z]
    
    # Determine a good threshold for visibility
    vmin = volume_for_3d_vis.min()
    vmax = volume_for_3d_vis.max()
    isomin_threshold = vmin + 0.25 * (vmax - vmin)

    fig = go.Figure(data=go.Volume(
        x=x.flatten(),
        y=y.flatten(),
        z=z.flatten(),
        value=volume_for_3d_vis.flatten(),
        isomin=isomin_threshold,
        isomax=vmax,
        opacity=0.1, # needs to be small to see through all surfaces
        surface_count=17, # needs to be a large number for good volume rendering
        caps=dict(x_show=False, y_show=False, z_show=False), # hide slice caps
        ))
    
    fig.update_layout(scene_xaxis_showticklabels=False,
                      scene_yaxis_showticklabels=False,
                      scene_zaxis_showticklabels=False,
                      title_text='Interactive 3D Volume Rendering',
                      margin=dict(l=0, r=0, b=0, t=40) # tight layout
                     )
    
    fig.show()
else:
    print("Cannot create 3D plot, no data available.")